# YOLOv8 人數辨識模型訓練（Google Colab 版本）

本 Notebook 用於在 Google Colab 上訓練 YOLOv8 模型，使用 GPU 加速訓練。

## ⚠️ 重要：Colab 無法直接訪問本地資料夾

**Colab 運行在遠端伺服器上，無法直接訪問你本地電腦的檔案！**

你必須先將檔案上傳到 Colab，有兩種方式：
1. **方法 1**：上傳 zip 檔案（適合小檔案）
2. **方法 2**：使用 Google Drive（推薦，適合大檔案）

## 使用步驟

1. **啟用 GPU**：在 Colab 中，點選 `執行階段` → `變更執行階段類型` → 選擇 `GPU`（T4 或 V100）
2. **上傳資料**：選擇方法 1 或方法 2 上傳 `yolo_training` 資料夾
3. **執行 Cells**：按順序執行所有 cells

## 注意事項

- Colab 免費版 GPU 使用時間有限（約 12 小時）
- 建議定期儲存檢查點（每 10 epochs 自動儲存）
- 訓練完成後記得下載模型和結果

In [1]:
# 安裝依賴套件
%pip install ultralytics>=8.0.0 opencv-python>=4.5.0 numpy>=1.24.0 pillow>=9.0.0 matplotlib>=3.5.0 pandas>=1.3.0 tables>=3.7.0 pyyaml -q
print("✓ 依賴套件安裝完成")

✓ 依賴套件安裝完成


In [2]:
# 檢查 GPU 是否可用
import torch

print("="*60)
print("GPU 檢查")
print("="*60)
print(f"CUDA 可用: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU 型號: {torch.cuda.get_device_name(0)}")
    print(f"GPU 記憶體: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"CUDA 版本: {torch.version.cuda}")
    print("✓ GPU 已啟用，將使用 GPU 進行訓練")
else:
    print("⚠ 警告: 未偵測到 GPU，將使用 CPU 訓練（速度較慢）")
    print("請確認已在 Colab 中啟用 GPU：執行階段 → 變更執行階段類型 → GPU")
print("="*60)

GPU 檢查
CUDA 可用: True
GPU 型號: Tesla T4
GPU 記憶體: 14.74 GB
CUDA 版本: 12.6
✓ GPU 已啟用，將使用 GPU 進行訓練


In [3]:
# 導入必要的庫
import os
import sys
import yaml
import tempfile
from pathlib import Path
from ultralytics import YOLO

print("✓ 庫導入完成")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✓ 庫導入完成


## ⚠️ 重要：上傳資料檔案

**Colab 無法直接訪問本地資料夾！** 請先上傳必要的檔案。

### 方法 1：使用檔案上傳（適合小檔案）

執行下面的 cell 來上傳 `yolo_training` 資料夾的 zip 檔案。

### 方法 2：使用 Google Drive（推薦，適合大檔案）

如果資料集很大，建議先上傳到 Google Drive，然後掛載 Drive。

In [ ]:
# ============================================
# 方法 1: 上傳 zip 檔案並解壓縮
# ============================================
# 步驟：
# 1. 在本地將 yolo_training 資料夾壓縮成 zip
# 2. 執行這個 cell 上傳並解壓縮

from google.colab import files
import zipfile

print("請上傳 yolo_training.zip 檔案...")
print("（在本地先將 yolo_training 資料夾壓縮成 zip）")
uploaded = files.upload()

# 解壓縮
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"\n正在解壓縮: {filename}")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('/content')
        print(f"✓ 解壓縮完成")
        
        # 設定工作目錄
        WORK_DIR = Path('/content/yolo_training')
        if WORK_DIR.exists():
            print(f"✓ 找到工作目錄: {WORK_DIR}")
        else:
            # 嘗試尋找解壓縮後的目錄
            possible_dirs = list(Path('/content').glob('*yolo*'))
            if possible_dirs:
                WORK_DIR = possible_dirs[0]
                print(f"✓ 找到工作目錄: {WORK_DIR}")
            else:
                print("✗ 錯誤: 找不到 yolo_training 目錄")
                print("請確認 zip 檔案中包含 yolo_training 資料夾")
        break
else:
    print("⚠ 未找到 zip 檔案，請使用方法 2（Google Drive）或重新上傳")

FileNotFoundError: [Errno 2] No such file or directory: 'yolo_training'

## 方法 2：使用 Google Drive（推薦）

如果資料集很大，建議使用 Google Drive。

In [ ]:
# ============================================
# 方法 2: 使用 Google Drive（如果使用方法 1，可以跳過這個 cell）
# ============================================
from google.colab import drive

# 掛載 Google Drive
print("正在掛載 Google Drive...")
drive.mount('/content/drive')
print("✓ Google Drive 已掛載")

# 設定工作目錄（根據你的實際路徑修改）
# 假設你將 yolo_training 資料夾放在 Drive 的根目錄
WORK_DIR = Path('/content/drive/MyDrive/thermo-presence/yolo_training')

# 如果路徑不同，請修改上面的路徑
# 例如：WORK_DIR = Path('/content/drive/MyDrive/你的資料夾/yolo_training')

if WORK_DIR.exists():
    print(f"✓ 找到工作目錄: {WORK_DIR}")
else:
    print(f"✗ 找不到工作目錄: {WORK_DIR}")
    print("\n請確認路徑是否正確，或手動設定 WORK_DIR")
    print("例如：WORK_DIR = Path('/content/drive/MyDrive/你的路徑/yolo_training')")

In [ ]:
# 確認工作目錄並切換
if 'WORK_DIR' not in locals() or not WORK_DIR.exists():
    print("⚠ 錯誤: 工作目錄未設定或不存在")
    print("\n請執行以下其中一個方法：")
    print("1. 執行「方法 1：上傳 zip 檔案」的 cell")
    print("2. 執行「方法 2：使用 Google Drive」的 cell")
    print("\n或者手動設定 WORK_DIR：")
    print("WORK_DIR = Path('/content/yolo_training')  # 根據實際路徑修改")
else:
    # 切換到工作目錄
    os.chdir(WORK_DIR)
    sys.path.insert(0, str(WORK_DIR))
    
    print("="*60)
    print("工作目錄設定")
    print("="*60)
    print(f"工作目錄: {os.getcwd()}")
    print(f"檔案列表:")
    for item in sorted(Path('.').iterdir()):
        if item.is_dir():
            print(f"  📁 {item.name}/")
        else:
            print(f"  📄 {item.name}")
    print("="*60)

In [ ]:
# 訓練參數設定（參考論文最佳配置）
MODEL_NAME = "yolov8n.pt"  # 輕量級模型，適合 edge device
IMAGE_SIZE = (192, 256)  # (height, width) 保持 aspect ratio
EPOCHS = 200
BATCH_SIZE = 32  # 在 GPU 上可以增加到 64 或 128 以加快訓練
LEARNING_RATE = 0.001
OPTIMIZER = "Adam"

# 資料增強參數
AUGMENT_SCALE = 0.5  # scaling range
AUGMENT_FLIP = 0.5  # horizontal flip probability
AUGMENT_MOSAIC = 1.0  # mosaic augmentation probability
AUGMENT_MIXUP = 0.1  # mixup augmentation probability

# 路徑設定
DATA_YAML = WORK_DIR / "data.yaml"
OUTPUT_DIR = WORK_DIR / "runs"
DATASET_DIR = WORK_DIR / "dataset"

print("="*60)
print("訓練參數設定")
print("="*60)
print(f"模型: {MODEL_NAME}")
print(f"影像尺寸: {IMAGE_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Optimizer: {OPTIMIZER}")
print("="*60)

In [ ]:
# 檢查資料集和配置檔案
print("="*60)
print("資料集檢查")
print("="*60)

# 檢查 data.yaml
if DATA_YAML.exists():
    print(f"✓ 找到資料配置檔案: {DATA_YAML}")
else:
    print(f"✗ 錯誤: 找不到資料配置檔案 {DATA_YAML}")
    print("請確認 data.yaml 已正確上傳")

# 檢查資料集目錄
if DATASET_DIR.exists():
    train_images = len(list((DATASET_DIR / 'images' / 'train').glob('*.png')))
    val_images = len(list((DATASET_DIR / 'images' / 'val').glob('*.png')))
    test_images = len(list((DATASET_DIR / 'images' / 'test').glob('*.png')))
    
    print(f"✓ 找到資料集目錄: {DATASET_DIR}")
    print(f"  - Train images: {train_images}")
    print(f"  - Val images: {val_images}")
    print(f"  - Test images: {test_images}")
else:
    print(f"✗ 錯誤: 找不到資料集目錄 {DATASET_DIR}")
    print("請確認 dataset 資料夾已正確上傳")

print("="*60)

## 開始訓練

執行以下 cell 開始訓練。訓練過程可能需要數小時，請耐心等待。

In [ ]:
# 更新 data.yaml 中的 path 為絕對路徑
if not DATA_YAML.exists() or not DATASET_DIR.exists():
    print("錯誤: 請先確認資料集和配置檔案已正確上傳")
else:
    with open(DATA_YAML, 'r', encoding='utf-8') as f:
        data_config = yaml.safe_load(f)
    
    # 將 path 設為絕對路徑
    data_config['path'] = str(DATASET_DIR.resolve())
    
    # 創建臨時的 YAML 檔案（避免修改原始檔案）
    temp_yaml = tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False, encoding='utf-8')
    yaml.dump(data_config, temp_yaml, default_flow_style=False, allow_unicode=True, sort_keys=False)
    temp_yaml.close()
    temp_yaml_path = temp_yaml.name
    
    print(f"✓ 資料配置檔案已更新")
    print(f"  資料集路徑: {DATASET_DIR.resolve()}")
    print(f"  臨時配置檔案: {temp_yaml_path}")

In [ ]:
# 載入模型並開始訓練
print("="*60)
print("YOLOv8 人數辨識模型訓練")
print("="*60)
print(f"模型: {MODEL_NAME}")
print(f"影像尺寸: {IMAGE_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Optimizer: {OPTIMIZER}")
print("="*60)

# 載入模型（YOLO 會自動使用 GPU）
print(f"\n載入模型: {MODEL_NAME}")
model = YOLO(MODEL_NAME)

# 訓練參數
train_args = {
    "data": temp_yaml_path,  # 使用包含絕對路徑的臨時 YAML 檔案
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "imgsz": IMAGE_SIZE[0],  # YOLO 使用單一尺寸（高度）
    "lr0": LEARNING_RATE,
    "optimizer": OPTIMIZER.lower(),
    "project": str(OUTPUT_DIR),
    "name": "train",
    "exist_ok": True,
    "save": True,
    "save_period": 10,  # 每 10 epochs 儲存一次
    "val": True,  # 啟用驗證
    "plots": True,  # 生成訓練圖表
    "rect": True,  # 啟用 Rectangular Training（避免將 192×256 墊補成 256×256）
    "augment": True,  # 啟用資料增強
    "hsv_h": 0.015,  # HSV-Hue augmentation
    "hsv_s": 0.7,  # HSV-Saturation augmentation
    "hsv_v": 0.4,  # HSV-Value augmentation
    "degrees": 0.0,  # 旋轉角度（熱影像不建議旋轉）
    "translate": 0.1,  # 平移
    "scale": AUGMENT_SCALE,  # 縮放
    "flipud": 0.0,  # 垂直翻轉（不適用）
    "fliplr": AUGMENT_FLIP,  # 水平翻轉
    "mosaic": AUGMENT_MOSAIC,  # Mosaic augmentation
    "mixup": AUGMENT_MIXUP,  # Mixup augmentation
    "copy_paste": 0.0,  # Copy-paste augmentation
    "device": 0 if torch.cuda.is_available() else "cpu",  # 明確指定使用 GPU 0
}

# 開始訓練
print("\n開始訓練...")
print("⚠ 注意: Colab 的免費 GPU 有使用時間限制（約 12 小時）")
print("建議定期儲存檢查點（每 10 epochs 自動儲存）\n")

try:
    results = model.train(**train_args)
    
    print("\n" + "="*60)
    print("訓練完成！")
    print("="*60)
    print(f"最佳模型: {OUTPUT_DIR / 'train' / 'weights' / 'best.pt'}")
    print(f"最後模型: {OUTPUT_DIR / 'train' / 'weights' / 'last.pt'}")
    print(f"訓練結果: {OUTPUT_DIR / 'train'}")
    print("="*60)
    
    # 顯示訓練結果摘要
    if hasattr(results, 'results_dict'):
        print("\n訓練結果摘要:")
        for key, value in results.results_dict.items():
            print(f"  {key}: {value}")
    
except Exception as e:
    print(f"\n訓練過程中發生錯誤: {e}")
    import traceback
    traceback.print_exc()
finally:
    # 清理臨時檔案
    try:
        if 'temp_yaml_path' in locals():
            os.unlink(temp_yaml_path)
            print(f"\n✓ 已清理臨時檔案: {temp_yaml_path}")
    except:
        pass

## 驗證模型（可選）

訓練完成後，可以在測試集上驗證模型性能。

## 🚀 快速測試（使用當前訓練的模型）

如果訓練還沒完成但需要測試，可以使用當前已保存的模型（best.pt 或 last.pt）進行快速驗證。

In [ ]:
# 快速測試當前模型（即使訓練還沒完成）
import os
from pathlib import Path
from ultralytics import YOLO
import yaml
import tempfile

# 檢查可用的模型
weights_dir = OUTPUT_DIR / "train" / "weights"
if weights_dir.exists():
    print("="*60)
    print("可用的模型檔案:")
    print("="*60)
    models = list(weights_dir.glob("*.pt"))
    for model_file in sorted(models):
        size_mb = model_file.stat().st_size / (1024 * 1024)
        print(f"  📦 {model_file.name} ({size_mb:.1f} MB)")
    print("="*60)
    
    # 使用最佳模型
    best_model = weights_dir / "best.pt"
    last_model = weights_dir / "last.pt"
    
    # 選擇要使用的模型
    model_to_use = best_model if best_model.exists() else last_model
    
    if model_to_use.exists():
        print(f"\n✓ 使用模型: {model_to_use.name}")
        model = YOLO(str(model_to_use))
        
        # 更新 data.yaml 路徑
        with open(DATA_YAML, 'r', encoding='utf-8') as f:
            data_config = yaml.safe_load(f)
        data_config['path'] = str(DATASET_DIR.resolve())
        
        temp_yaml = tempfile.NamedTemporaryFile(mode='w', suffix='.yaml', delete=False, encoding='utf-8')
        yaml.dump(data_config, temp_yaml, default_flow_style=False, allow_unicode=True, sort_keys=False)
        temp_yaml.close()
        temp_yaml_path = temp_yaml.name
        
        try:
            # 在測試集上快速驗證
            print("\n在測試集上快速驗證...")
            results = model.val(data=temp_yaml_path, split="test", plots=True)
            
            print("\n" + "="*60)
            print("測試集結果摘要")
            print("="*60)
            if hasattr(results, 'results_dict'):
                for key, value in results.results_dict.items():
                    if isinstance(value, (int, float)):
                        print(f"  {key}: {value:.4f}")
                    else:
                        print(f"  {key}: {value}")
            
            print("\n" + "="*60)
            print("✓ 快速測試完成！")
            print(f"詳細圖表保存在: {OUTPUT_DIR / 'val'}")
            print("="*60)
        except Exception as e:
            print(f"\n測試錯誤: {e}")
            import traceback
            traceback.print_exc()
        finally:
            os.unlink(temp_yaml_path)
    else:
        print("✗ 找不到可用的模型檔案")
else:
    print("✗ 找不到模型目錄，請先開始訓練")

In [ ]:
# 可選：測試單張影像（視覺化檢查）
# 取消註解並修改影像路徑來測試單張影像

# test_image_path = DATASET_DIR / "images" / "test" / "008__13_26_20_000000.png"
# if test_image_path.exists() and 'model' in locals():
#     print(f"測試影像: {test_image_path}")
#     results = model.predict(
#         source=str(test_image_path),
#         conf=0.25,
#         save=True,
#         project=str(OUTPUT_DIR),
#         name="single_test"
#     )
#     print(f"\n✓ 檢測到 {len(results[0].boxes)} 個人")
#     print(f"結果保存在: {OUTPUT_DIR / 'single_test'}")
# else:
#     print("請先執行上面的快速測試 cell，或修改 test_image_path 為實際的影像路徑")

In [ ]:
# 在測試集上驗證模型
model_path = OUTPUT_DIR / "train" / "weights" / "best.pt"

if model_path.exists():
    print(f"載入最佳模型: {model_path}")
    model = YOLO(str(model_path))
    
    # 在測試集上驗證
    print("在測試集上驗證...")
    results = model.val(data=str(DATA_YAML), split="test")
    
    print("\n驗證結果:")
    if hasattr(results, 'results_dict'):
        for key, value in results.results_dict.items():
            print(f"  {key}: {value}")
else:
    print(f"錯誤: 找不到模型檔案 {model_path}")
    print("請先完成訓練")

## 下載訓練結果

訓練完成後，可以下載模型和訓練結果到本地電腦。

In [ ]:
# 下載訓練結果（Colab 專用）
from google.colab import files
import zipfile

results_dir = OUTPUT_DIR / "train"
if results_dir.exists():
    zip_path = WORK_DIR / "training_results.zip"
    
    print("正在壓縮訓練結果...")
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for file_path in results_dir.rglob('*'):
            if file_path.is_file():
                # 保持相對路徑結構
                arcname = file_path.relative_to(results_dir.parent)
                zipf.write(file_path, arcname)
    
    print(f"✓ 訓練結果已壓縮: {zip_path}")
    print("正在下載...")
    
    # 下載
    files.download(str(zip_path))
    print("✓ 下載完成！")
    
    # 也可以單獨下載最佳模型
    best_model = results_dir / "weights" / "best.pt"
    if best_model.exists():
        print(f"\n單獨下載最佳模型: {best_model}")
        files.download(str(best_model))
else:
    print("找不到訓練結果目錄，請先完成訓練")

## 使用說明

### 在 Google Colab 中使用：

1. **上傳 Notebook**：
   - 將 `train_colab.ipynb` 上傳到 Google Colab
   - 或直接在 Colab 中建立新 notebook 並複製 cells

2. **上傳資料集**：
   - 將整個 `yolo_training` 資料夾上傳到 Colab
   - 或使用 Google Drive 掛載（推薦，適合大檔案）

3. **啟用 GPU**：
   - 點選 `執行階段` → `變更執行階段類型`
   - 選擇 `GPU`（T4 或 V100）

4. **執行 Cells**：
   - 按順序執行所有 cells
   - 訓練過程可能需要數小時

5. **下載結果**：
   - 訓練完成後執行下載 cell
   - 或從 Colab 左側檔案瀏覽器下載

### 在 VS Code 中使用 Colab Extension：

1. 安裝 "Google Colab" extension
2. 開啟 `train_colab.ipynb`
3. 點選右上角的 Colab 圖示連接到 Colab
4. 上傳必要的檔案到 Colab
5. 執行 cells